# 00 — Kiểm tra môi trường & rủi ro kỹ thuật cao nhất: cài `mamba-ssm` trên Colab T4

**Chạy notebook này đầu tiên, trên Google Colab với Runtime = GPU (T4).**

Mục tiêu (Tuần 1–2 theo đề cương): xác nhận sớm nhất có thể liệu `mamba-ssm`
có build/chạy được trên T4 hay không, và nếu không, xác nhận phương án dự
phòng (Mamba thuần PyTorch, không cần CUDA kernel tùy biến) hoạt động đúng.

Đây là rủi ro cao nhất của toàn bộ đề tài — nếu mục 3 thất bại hoàn toàn,
cần báo GVHD sớm để điều chỉnh phạm vi.

**Trạng thái (xem chi tiết đầy đủ ở `docs/notes/mamba_ssm_install_log.md`):**
Lần 1 (2026-09-14, torch 2.10.0+cu128) — Cách 1 (`pip install mamba-ssm`
thẳng) **thất bại** ngay ở bước lấy metadata ("Getting requirements to
build wheel"), trước cả khi build CUDA kernel. Nguyên nhân khả dĩ: pip build
trong môi trường cô lập (PEP 517) không thấy torch đã cài sẵn trên Colab, và
torch 2.10+cu128 còn quá mới để có wheel prebuilt. Notebook đã cập nhật để
thử Cách 2 (`--no-build-isolation`, build từ source) làm ưu tiên chính.

## 1. Kiểm tra GPU & phiên bản torch/CUDA

In [ ]:
!nvidia-smi

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version (torch build):", torch.version.cuda)
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("compute capability:", torch.cuda.get_device_capability(0))

import sys
print("python:", sys.version)

## 2. Cài `mamba-ssm`

**Cách 1 (`pip install mamba-ssm` thẳng) đã xác nhận THẤT BẠI trên Colab —
xem log. Bỏ qua, chạy thẳng Cách 2 bên dưới.**

Cách 2 build từ source với `--no-build-isolation` để setup.py nhìn thấy
torch đã cài sẵn trên Colab (thay vì môi trường build cô lập không có gì).
Nếu vẫn lỗi, cell sau in ra **toàn bộ log lỗi thật** (không bị Colab thu gọn)
để chẩn đoán tiếp — copy log đó vào `docs/notes/mamba_ssm_install_log.md`.

In [ ]:
# Cách 1 — đã thử, THẤT BẠI (xem docs/notes/mamba_ssm_install_log.md, Lần 1).
# Giữ lại đây để tham khảo, không cần chạy lại trừ khi muốn kiểm tra lại từ đầu.

# !pip install -q mamba-ssm causal-conv1d>=1.4.0

In [ ]:
# Cách 2 (ưu tiên chính): cài từ source, tắt build isolation để dùng đúng
# torch đã cài sẵn trên Colab. Dùng subprocess thay vì !pip để bắt được full
# log lỗi (Colab hay thu gọn output dài của lệnh !pip khi lỗi).
import subprocess

def run(cmd):
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print("--- STDERR (tail) ---")
        print(result.stderr[-4000:])
    print("return code:", result.returncode)
    return result.returncode == 0

import sys

ok = run([sys.executable, "-m", "pip", "install", "-q", "packaging", "ninja"])
ok = ok and run([
    sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation",
    "git+https://github.com/Dao-AILab/causal-conv1d",
])
ok = ok and run([
    sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation",
    "git+https://github.com/state-spaces/mamba",
])
print("\nCách 2 thành công:" if ok else "\nCách 2 vẫn thất bại — xem STDERR ở trên, ghi lại vào docs/notes/mamba_ssm_install_log.md, Lần 2.", ok)

In [ ]:
# Kiểm tra import + forward pass nhỏ với CUDA kernel thật
try:
    from mamba_ssm import Mamba
    m = Mamba(d_model=64, d_state=16, d_conv=4, expand=2).to("cuda")
    x = torch.randn(2, 32, 64, device="cuda")
    y = m(x)
    print("OK — mamba-ssm (CUDA kernel) hoạt động. Output shape:", y.shape)
    MAMBA_CUDA_OK = True
except Exception as e:
    print("THẤT BẠI — mamba-ssm CUDA kernel không chạy được:")
    print(repr(e))
    MAMBA_CUDA_OK = False

## 3. Phương án dự phòng: Mamba thuần PyTorch (không cần CUDA kernel)

`mamba_ssm` cung cấp `selective_scan_ref` — cài đặt tham chiếu thuần PyTorch,
**chậm hơn** nhưng đúng về mặt số học và không cần biên dịch bất kỳ CUDA
kernel nào. Đây là phương án dự phòng nêu trong đề cương nếu mục 2 thất bại
hoàn toàn trên T4. **Chỉ có ý nghĩa nếu mục 2 đã cài được package** (dù CUDA
kernel không chạy) — nếu package chưa cài được gì cả thì ô này cũng sẽ báo
lỗi tương tự, không nói lên được gì về fallback thật sự (đã xảy ra ở Lần 1).

In [ ]:
try:
    from mamba_ssm.ops.selective_scan_interface import selective_scan_ref
    print("OK — selective_scan_ref (thuần PyTorch) import được — có thể dùng làm fallback.")
    FALLBACK_OK = True
except Exception as e:
    print("selective_scan_ref cũng không import được:")
    print(repr(e))
    print("=> Cần cân nhắc phương án khác (vd. cài đặt S4/S6 tối giản từ đầu).")
    FALLBACK_OK = False

## 4. Kết luận rủi ro (ghi lại kết quả để báo cáo GVHD)

In [ ]:
print(f"mamba-ssm CUDA kernel hoạt động trên T4: {MAMBA_CUDA_OK}")
print(f"Fallback thuần PyTorch khả dụng:        {FALLBACK_OK}")
if MAMBA_CUDA_OK:
    print("=> Dùng CUDA kernel thật cho tốc độ train/inference tốt nhất.")
elif FALLBACK_OK:
    print("=> Dùng selective_scan_ref (chậm hơn) — vẫn tiếp tục đề tài đúng kế hoạch,")
    print("   nhưng cần ghi rõ trong báo cáo (Chương 2) và ước lượng lại thời gian train.")
else:
    print("=> RỦI RO NGHIÊM TRỌNG — báo GVHD ngay để điều chỉnh phạm vi đề tài.")

print("\n=> Nhớ copy kết quả (thành công/thất bại + log lỗi nếu có) vào")
print("   docs/notes/mamba_ssm_install_log.md trước khi đóng Colab.")

## 5. Cài các thư viện còn lại của pipeline

In [ ]:
!pip install -q librosa soundfile sentencepiece tokenizers datasets jiwer gradio

## 6. (Tuần 3) Thử tải thử VietSuperSpeech — chỉ kiểm tra kết nối, chưa tải toàn bộ

In [ ]:
from datasets import load_dataset

# streaming=True để chỉ xem thử vài mẫu, không tải 267 giờ audio ngay bây giờ
ds = load_dataset("thanhnew2001/VietSuperSpeech", split="train", streaming=True)
sample = next(iter(ds))
print(sample.keys())
print({k: v for k, v in sample.items() if k != "audio"})